In [ ]:
#step1: Loading the libraries and starting the spark session
from pyspark.sql import SparkSession
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, Normalizer, StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
# Create a SparkSession
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()

# Suppress unnecessary logging
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Step 2: Load the dataset and preprocess the data

# Loading the dataset
data_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json" 
full_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviewscombined.json"
stopwords_path = "../data/stopwords.txt"

# Loading stopwords into a set
with open(stopwords_path, "r") as f:
    stopwords = set(line.strip() for line in f)
stopwords.add("")

# Loading the dataset into a Spark DataFrame
dataframe = spark.read.json(data_path)

# 2.1 Preprocessing the data
# Tokenization
tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern="\s+|[,.;]", minTokenLength=2)
dataframe = tokenizer.transform(dataframe)

# Stopword removal
stop_words_remover = StopWordsRemover(inputCol="Words", outputCol="filtered_words", stopWords=list(stopwords))
dataframe = stop_words_remover.transform(dataframe)

# TF-IDF calculation
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[hashingTF, idf])
pipeline_model = pipeline.fit(dataframe)
dataframe = pipeline_model.transform(dataframe)

# Display the preprocessed dataframe schema and the first few rows
dataframe.printSchema()
dataframe.show(5)

In [ ]:
# Step 3: Splitting the dataset into training, validation, and test sets
# Defining proportions for splitting the dataset
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Split the dataset into training, validation, and test sets
train_data, val_data, test_data = dataframe.randomSplit([train_ratio, val_ratio, test_ratio], seed=42)

# Cache the datasets for better performance
train_data.cache()
val_data.cache()
test_data.cache()

# Print the sizes of the datasets
print("Training set size:", train_data.count())
print("Validation set size:", val_data.count())
print("Test set size:", test_data.count())

In [ ]:
# Step 4: Applying vector length normalization using Normalizer with L2 norm
# Create a Normalizer transformer
normalizer = Normalizer(inputCol="features", outputCol="normalized_features", p=2.0)

# Applying normalization to training, validation, and test data
train_data = normalizer.transform(train_data)
val_data = normalizer.transform(val_data)
test_data = normalizer.transform(test_data)

# Displaying the first few rows of the normalized features in the training set
train_data.select("normalized_features").show(5, truncate=False)

In [ ]:
# Step 5: Preparing the SVM classifier within a pipeline

# Create a StringIndexer transformer
label_indexer = StringIndexer(inputCol="category", outputCol="label")

# Updating the pipeline to include the StringIndexer
pipeline = Pipeline(stages=[label_indexer])

# Fitting the pipeline to the training data
pipeline_model = pipeline.fit(train_data)
train_data_transformed = pipeline_model.transform(train_data)

In [ ]:
# Step 6: Setting up the OneVsRest SVM model
svm = LinearSVC(maxIter=10, regParam=0.1)
ovr = OneVsRest(classifier=svm)


In [ ]:
# Step 7: Setting up the parameter grid for grid search
paramGrid = ParamGridBuilder() \
    .addGrid(svm.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(svm.maxIter, [10, 50]) \
    .addGrid(svm.standardization, [True, False]) \
    .addGrid(svm.numFeatures, [100, 500]) \
    .build()

In [ ]:
# Step 8: Setting up cross-validation
evaluator = MulticlassClassificationEvaluator(metricName="f1")

crossval = CrossValidator(estimator=ovr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)  # Use 3-fold cross-validation

In [ ]:
# Step 9: Training the model using cross-validation
cvModel = crossval.fit(train_data_transformed)

In [ ]:
# Step 10: Evaluating the model on the test set
test_data_transformed = pipeline_model.transform(test_data)
predictions = cvModel.transform(test_data_transformed)

f1_score = evaluator.evaluate(predictions)
print("Test set F1 score:", f1_score)

In [ ]:
spark.stop()